# ch05 Bonus 04：训练加速（Training Speed）

> 对照官方 `ch05/10_llm-training-speed`

## 一句话

用 **混合精度（autocast）+ 梯度累积 + torch.compile** 三板斧，在不改模型的前提下大幅提升训练吞吐。

## 三大加速手段

| 手段 | 原理 | 收益 |
|------|------|------|
| **混合精度 (autocast)** | 前向用 fp16/bf16，省一半显存和算力 | ~2× 速度 |
| **梯度累积** | 多个小 batch 的梯度累加再更新，等效大 batch | 突破显存限制 |
| **torch.compile** | 编译计算图，融合 kernel | ~1.3-2× 速度 |

> 混合精度会让部分计算精度降低，但 LLM 训练对此不敏感（尤其 bf16）。

In [ ]:
import time
import torch
import torch.nn.functional as F
from pathlib import Path
from src.gpt import GPTModel, GPT_CONFIG_124M, create_dataloader_v1

cfg = dict(GPT_CONFIG_124M)
cfg.update({"emb_dim": 256, "n_layers": 4, "n_heads": 4, "context_length": 256})
text = Path("data/the-verdict.txt").read_text(encoding="utf-8")
dl = create_dataloader_v1(text, batch_size=2, max_length=cfg["context_length"],
                          stride=cfg["context_length"], shuffle=True, drop_last=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def benchmark(use_amp=False, n_batches=6):
    """对比 fp32 vs 混合精度的单步训练耗时。"""
    torch.manual_seed(123)
    model = GPTModel(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    model.train()
    it = iter(dl)
    # 预热 1 步
    x, y = next(it)
    start = time.perf_counter()
    for _ in range(n_batches):
        x, y = next(it)
        opt.zero_grad()
        ctx = torch.autocast("cuda", dtype=torch.bfloat16) if (use_amp and device.type=="cuda") else torch.cuda.amp.autocast(enabled=False)
        with ctx:
            loss = F.cross_entropy(model(x.to(device)).flatten(0,1), y.to(device).flatten())
        loss.backward(); opt.step()
    if device.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - start) / n_batches

In [ ]:
# 对比基准（CPU 上混合精度收益有限，GPU 上更明显）
print(f"设备: {device}")
try:
    t_fp32 = benchmark(use_amp=False)
    t_amp = benchmark(use_amp=True)
    print(f"fp32 平均每步: {t_fp32*1000:.1f} ms")
    if device.type == "cuda":
        print(f"混合精度每步: {t_amp*1000:.1f} ms")
        print(f"加速比: {t_fp32/t_amp:.2f}×")
    else:
        print("(CPU 上混合精度加速有限，GPU 上收益更明显)")
except Exception as e:
    print(f"基准测试异常: {e}")

In [ ]:
# 梯度累积演示：等效大 batch
accumulation_steps = 4  # 累积 4 个小 batch = 等效 batch_size=8
torch.manual_seed(123)
model = GPTModel(cfg).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
model.train()
it = iter(dl)

opt.zero_grad()
for i in range(accumulation_steps):
    x, y = next(it)
    loss = F.cross_entropy(model(x.to(device)).flatten(0,1), y.to(device).flatten())
    # 关键：除以累积步数，使累积后等效于一次大 batch
    (loss / accumulation_steps).backward()
    print(f"  累积步 {i+1}: 单步 loss={loss.item():.4f}")
opt.step()  # 累积完毕才真正更新一次
print("\n💡 显存只够装 batch=2，但通过累积 4 次等效实现了 batch=8 的训练。")